(sequence-search-sort)=

# Search Contracts and Repeated Queries


```{contents}
:local:
:depth: 2
```


```{index} Specify the Answer Before Counting Comparisons
```



```{index} Specify the Answer Before Counting Comparisons
```

## Specify the Answer Before Counting Comparisons



A roster contains codes A, C, C, F. Searching for C could mean finding any occurrence, the first position, every matching position, or simply whether it exists. Those are different result contracts. A faster implementation of a weaker task is not automatically a better solution to the original task.

A first-match linear search examines positions in arrival order. Before examining position i, the invariant is that no earlier position matches. On C, it visits A and then the first C, returning position one. On a missing Z, it reaches the end, and the empty unexamined suffix establishes that no position matched. Empty input reaches that conclusion without reading any element.

| Target | Positions visited by first-match scan | Result |
| --- | --- | --- |
| A | 0 | 0 |
| C | 0, 1 | 1 |
| Z | 0, 1, 2, 3 | Missing |

Binary search answers a question using an ordered region. Each comparison rules out part of the search interval only because the ordering policy is consistent with the ordering used to prepare the data. A visually sorted report using one policy cannot safely be searched using another. Also do not return the position in a sorted snapshot as though it were the original arrival position.

For related explanation, see the further reading below.[^development-reading]


```{index} Trace a Shrinking Candidate Region
```



```{index} Trace a Shrinking Candidate Region
```

## Trace a Shrinking Candidate Region



For a half-open interval [lo, hi), every candidate is at least lo and strictly before hi. Empty means lo equals hi. A midpoint comparison removes a region that cannot contain the desired result under the contract. The important progress argument is that every continuing iteration strictly shrinks the candidate region. Merely changing lo or hi is not enough if an update can leave the same interval.

A common off-by-one mistake mixes an inclusive upper bound with a loop intended for half-open intervals. Trace an empty array, one element, and a missing target outside each end before celebrating a successful middle match. For duplicates, an ordinary any-match binary search need not return the first occurrence; Chapter 23 develops lower and upper bounds for that stronger query.

Guided workload decision: a report is prepared once and searched one thousand times. List the preparation work as well as the query work. Now suppose a new record arrives after every query. Explain which assumption changed and why the original recommendation may need revision. The following preserved workload material develops that comparison without equating one quick benchmark with a universal threshold.


```{index} Searching Ordered and Unordered Data
```

<span id="searching-ordered-and-unordered-data"></span>


An inventory application must distinguish finding an item from ordering the entire catalog. A search contract states which answer you need: the first match, any match, all matches, or a missing result. Representation affects the cost of producing that answer.

Chapter 13 introduced comparison, equality, and operation counts; Chapter 14 introduced array and linked storage. Apply those ideas to a service desk looking for a ticket ID. Every example below owns its array and includes the complete search implementation. Empty input and missing targets are ordinary cases to test, not unusual afterthoughts.


```{index} First-Match Linear Search
```

## First-Match Linear Search




```{index} First-Match Linear Search
```


Linear search visits values in order and returns immediately when equality succeeds. It accepts unsorted data. Return the first matching index or -1; preserve the input and do not treat index zero as a missing result.

The processed prefix contains no match. When the loop reaches the end, every position has been checked, so -1 is justified. A match at index k needs k+1 equality checks. A missing target needs n checks; an empty array needs zero. This is O(n) worst-case work under the fixed-cost comparison model.


In [1]:
using System;
using System.Collections.Generic;

int[] ids = {42, 17, 42, 8};
Console.WriteLine(BasicSearch.LinearFirst(ids, 42, EqualityComparer<int>.Default));
Console.WriteLine(BasicSearch.LinearFirst(ids, 99, EqualityComparer<int>.Default));
Console.WriteLine(BasicSearch.LinearFirst(Array.Empty<int>(), 42, EqualityComparer<int>.Default));

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}


0
-1
-1


The results are 0, -1, and -1. An equality comparer makes the matching rule explicit. For symbolic IDs you might choose case-sensitive or case-insensitive string equality, but use the same policy consistently. Matching a customer's name is not necessarily the same as identifying the customer's record.

For linked nodes, linear traversal is also natural: follow each Next reference once. Repeatedly requesting an item by numeric index can rescan the list from its head and accidentally turn a linear task into quadratic work.


```{index} Binary Search Requires the Right Order
```

## Binary Search Requires the Right Order




```{index} Binary Search Requires the Right Order
```


(binarysearching)=

Binary search repeatedly discards half of a sorted, random-access sequence. The data must be sorted under the same ordering comparer used by the search. Sorting by price and then searching by product ID does not meet that precondition.

The interval [low, high) contains all remaining candidates. Start low at zero and high at Length. Compare the middle value: equality returns its index; a smaller middle excludes the left portion and the middle; a larger middle excludes the middle and the right portion. The interval shrinks on every unsuccessful comparison.

The safe midpoint expression low + (high - low) / 2 avoids adding two large indexes. High is a boundary, not always a valid element index. The method never indexes an empty interval.


In [1]:
using System;
using System.Collections.Generic;

int[] ids = {8, 17, 42, 42, 60};
Console.WriteLine(BasicSearch.BinaryAny(ids, 17, Comparer<int>.Default));
Console.WriteLine(BasicSearch.BinaryAny(ids, 99, Comparer<int>.Default));
Console.WriteLine(BasicSearch.BinaryAny(Array.Empty<int>(), 17, Comparer<int>.Default));

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}


1
-1
-1


There are at most floor(log2 n)+1 comparisons for a nonempty array, and O(1) auxiliary storage in this iterative version. The method returns any matching index: duplicate values do not imply the first matching index or the first arrival.

Trace a missing target 20 in [8,17,42,42,60]. The interval narrows until its boundaries meet between 17 and 42. Missing searches and singleton ranges are essential tests because an incorrect boundary update can skip candidates or fail to make progress.


```{index} Count the Comparisons of Binary Search
```

### Count the Comparisons of Binary Search

The bound floor(log2 n) + 1 comes from watching the interval. Each unsuccessful comparison discards at least half of the remaining candidates, so after k comparisons at most n / 2^k candidates remain. The search must stop once fewer than one candidate remains, which happens after about log2 n halvings. Doubling the array adds one halving and therefore at most one more comparison; a search over a million sorted records needs at most 20 comparisons, and over a billion at most 30. Linear search, by contrast, doubles its worst case whenever the array doubles.

| n | Longest possible search | Linear search, worst case |
| ---: | ---: | ---: |
| 1 | 1 | 1 |
| 7 | 3 | 7 |
| 8 | 4 | 8 |
| 1,000 | 10 | 1,000 |
| 1,000,000 | 20 | 1,000,000 |

The example counts comparisons on sorted arrays of those sizes for a present key near the end, a missing key beyond the end, and a key that happens to sit at the first midpoint, which is the best case.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 1, 7, 8, 1000, 1000000 })
{
    int[] sorted = Enumerable.Range(0, n).Select(i => i * 2).ToArray();   // even numbers 0, 2, 4, ...
    int firstMiddle = sorted[(n - 1) / 2 == 0 ? 0 : n / 2];
    Console.WriteLine($"n={n,8}: best={Count(sorted, firstMiddle),2}  last={Count(sorted, sorted[^1]),2}  missing={Count(sorted, -1),2}  bound={(int)Math.Floor(Math.Log2(n)) + 1,2}");
}

static int Count(int[] sorted, int target)
{
    int low = 0, high = sorted.Length, comparisons = 0;
    while (low < high)
    {
        int middle = low + (high - low) / 2;
        comparisons++;
        if (sorted[middle] == target) return comparisons;
        if (sorted[middle] < target) low = middle + 1;
        else high = middle;
    }
    return comparisons;
}

n=       1: best= 1  last= 1  missing= 1  bound= 1
n=       7: best= 1  last= 3  missing= 3  bound= 3
n=       8: best= 1  last= 3  missing= 4  bound= 4
n=    1000: best= 1  last= 9  missing=10  bound=10
n= 1000000: best= 1  last=19  missing=20  bound=20


Every count stays at or under the bound in the last column, and the best case is a single comparison whenever the target sits at the first midpoint. The missing key costs the full bound because the interval must shrink to empty before the method can say so. Compare the growth down the `last` column with the array sizes: multiplying n by a thousand added about ten comparisons, which is the logarithm doing its work. This is the reason a sorted snapshot can pay for itself when the same data is searched many times, the trade the next subsection weighs.

```{index} Code Tracing: Narrow the Interval
```

### Code Tracing: Narrow the Interval

Trace `BinaryAny` on the array `[8, 17, 42, 42, 60]` for the missing target 20. Before each comparison write `low`, `high`, `middle`, and the value compared; after it, write the new interval. Stop when `low == high` and state the result. Then repeat for target 42 and say which of the two 42s is returned.

In [ ]:
// Solution
using System;

int[] ids = { 8, 17, 42, 42, 60 };
Trace(ids, 20);
Trace(ids, 42);

static void Trace(int[] sorted, int target)
{
    Console.WriteLine($"target {target}:");
    int low = 0, high = sorted.Length;
    while (low < high)
    {
        int middle = low + (high - low) / 2;
        Console.Write($"  [{low}, {high}) middle={middle} value={sorted[middle]} -> ");
        if (sorted[middle] == target) { Console.WriteLine($"found at index {middle}"); return; }
        if (sorted[middle] < target) { low = middle + 1; Console.WriteLine($"go right, low={low}"); }
        else { high = middle; Console.WriteLine($"go left, high={high}"); }
    }
    Console.WriteLine($"  [{low}, {high}) is empty -> missing");
}

target 20:
  [0, 5) middle=2 value=42 -> go left, high=2
  [0, 2) middle=1 value=17 -> go right, low=2
  [2, 2) is empty -> missing
target 42:
  [0, 5) middle=2 value=42 -> found at index 2


For 20 the interval moves from [0, 5) to [0, 2) after comparing with 42, then to [2, 2) after comparing with 17, and the empty interval reports the key missing after two comparisons. For 42 the first midpoint is index 2, whose value is 42, so the search returns index 2 at once: the first of the two duplicates, but only because the midpoint happened to land there. With the array `[8, 42, 42, 42, 60]` the first midpoint would return index 2 again, which is now the middle duplicate. Chapter 23 builds the lower-bound search that always returns the first.

```{index} Code Completion: Finish the Boundary Updates
```

### Code Completion: Finish the Boundary Updates

The two boundary updates are missing. Complete them so that the interval stays half-open and shrinks on every unsuccessful comparison. The expected output is `index 3`, `missing`, `index 0`, and `missing` for the empty array. A wrong update that leaves `low` or `high` unchanged makes the loop run forever, so predict each update before running.

In [ ]:
using System;

int[] prices = { 5, 9, 12, 20, 31, 44 };
Report(prices, 20);
Report(prices, 21);
Report(prices, 5);
Report(Array.Empty<int>(), 5);

static void Report(int[] sorted, int target)
{
    int index = BinaryAny(sorted, target);
    Console.WriteLine(index >= 0 ? $"index {index}" : "missing");
}

static int BinaryAny(int[] sorted, int target)
{
    int low = 0, high = sorted.Length;
    while (low < high)
    {
        int middle = low + (high - low) / 2;
        if (sorted[middle] == target) return middle;
        // Your code starts here.
        // If the middle value is smaller than the target, discard it and everything before it.
        // Otherwise discard it and everything after it.

        // Your code ends here.
    }
    return -1;
}

In [ ]:
// Solution
using System;

int[] prices = { 5, 9, 12, 20, 31, 44 };
Report(prices, 20);
Report(prices, 21);
Report(prices, 5);
Report(Array.Empty<int>(), 5);

static void Report(int[] sorted, int target)
{
    int index = BinaryAny(sorted, target);
    Console.WriteLine(index >= 0 ? $"index {index}" : "missing");
}

static int BinaryAny(int[] sorted, int target)
{
    int low = 0, high = sorted.Length;
    while (low < high)
    {
        int middle = low + (high - low) / 2;
        if (sorted[middle] == target) return middle;
        if (sorted[middle] < target) low = middle + 1;   // middle is excluded: it was too small
        else high = middle;                               // middle is excluded: high is exclusive
    }
    return -1;
}

index 3
missing
index 0
missing


Both updates exclude `middle`, because the comparison already proved it is not the target. Writing `low = middle` instead of `middle + 1` stops making progress when the interval has two elements and the left one is too small: `middle` recomputes to the same index forever. Writing `high = middle - 1` is the mirror mistake for an inclusive-bound version; in this half-open version it would skip the element at `middle - 1`, and the search for 5 would then report it missing.

```{index} Library Results Have Their Own Contract
```

## Library Results Have Their Own Contract




```{index} Library Results Have Their Own Contract
```


Array.IndexOf uses -1 for a missing value. Array.BinarySearch returns a negative encoded insertion position instead; test result >= 0 before using it as an index. The bitwise complement ~result decodes the position where a missing value could be inserted while preserving order.

Do not mix a teaching method's sentinel with a library method's contract. Read the API promise and check boundaries before using a returned index. The required learning is the sign test; insertion-position encoding is an applied extension.


In [1]:
using System;
using System.Collections.Generic;

int[] ids = {8, 17, 42};
int found = Array.BinarySearch(ids, 20);
Console.WriteLine(found >= 0 ? "Found" : "Missing");
if (found < 0) Console.WriteLine($"Insertion position: {~found}");


Missing
Insertion position: 2


```{index} Choose a Search for the Workload
```

## Choose a Search for the Workload




```{index} Choose a Search for the Workload
```


One scan may be cheaper than sorting solely to answer one query. Repeated searches on static data can justify a sorted snapshot: sorting once has its own preparation cost, followed by logarithmic searches. Updates may invalidate that snapshot. A dictionary supports exact-key access but does not replace an ordered range query.

State the input size, required answer, update pattern, equality policy, and representation before choosing an algorithm. A stopwatch observation can support a particular measurement; it cannot establish a general growth bound. The next sections make sorting work visible and connect preparation to a complete reporting workload.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A table is sorted by customer name. A colleague runs binary search on it for a customer ID and gets a wrong answer with no exception. Why no exception?

```{dropdown} Answer
Binary search never checks that its precondition holds; it only compares the target with midpoints and discards halves. When the array is ordered by a different key, each discard is based on a false assumption, so the method confidently reports a wrong position or a missing key. The contract puts the ordering obligation on the caller.
```

2. How many comparisons can binary search need on a sorted array of 5,000 records, and how many on 10,000?

```{dropdown} Answer
At most 13 on 5,000, because 2 to the 12th is 4,096 and 2 to the 13th is 8,192, so floor(log2 5000) + 1 is 13. On 10,000 at most 14. Doubling the input added one comparison.
```

3. `Array.BinarySearch` returned -4. Was the key found, and what does the number tell you?

```{dropdown} Answer
Not found: a negative result means missing. The bitwise complement, `~(-4)`, is 3, the index at which the key could be inserted while keeping the array sorted. Code must test the sign before using the result as an index.
```

4. A report of 200 records is searched once a day. Should it be sorted first so the search is logarithmic?

```{dropdown} Answer
No. One linear scan costs at most 200 comparisons, while sorting first costs more than that before the single search even begins. Preparation pays off only when the same sorted data is searched many times between changes.
```

```{index} Practice: State the Matching Policy
```

## Practice: State the Matching Policy




```{index} Practice: State the Matching Policy
```


Find the first case-insensitive ticket ID without sorting or changing the input.


In [ ]:
using System;
using System.Collections.Generic;

string[] ids = {"T-100", "t-100", "T-200"};
int first = BasicSearch.LinearFirst(ids, "t-100", StringComparer.Ordinal /* TODO: choose case-insensitive equality */);
Console.WriteLine(first);

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}


In [1]:
using System;
using System.Collections.Generic;

string[] ids = {"T-100", "t-100", "T-200"};
int first = BasicSearch.LinearFirst(ids, "t-100", StringComparer.OrdinalIgnoreCase);
Console.WriteLine(first);

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}


0


Repeated queries may justify preparing a sorted snapshot, but preparation and later updates have a cost. The next section develops sorting and shows how to preserve arrival order while preparing a report. Choose between one scan and repeated searches only after including that work.


```{rubric} Footnotes
```


[^1601_searching-1]: Binary search needs compatible ordering and random access. More detailed duplicate-boundary searches and algorithm proofs remain in optional Part VI.


[^development-reading]: Further reading: [OpenDSA, Search in Sorted Arrays](https://opendsa-server.cs.vt.edu/OpenDSA/Books/Catalog/html/SortedSearch.html). The worked business scenarios and guided reasoning here are original teaching examples.
